# Step 5 — SVM Signal Peptide Classifier
**LB2 Project · Group 7 · Signal Peptide Prediction**

### Objective
Train a **Support Vector Machine (SVM)** classifier for signal peptide detection using
biochemical features extracted from the N-terminal region of each protein.

### Pipeline
1. Extract biochemical features from the N-terminal region of every sequence
2. Normalise features (z-score, fit on training data only)
3. Select the top features using Random Forest importance (5-fold CV)
4. Train RBF-kernel SVM with grid search over C and γ (optimising MCC)
5. Evaluate with 5-fold cross-validation
6. Retrain on full training set → blind benchmark evaluation
7. Compare: all features vs selected features

### Features extracted
| Feature group | Description | Region |
|---|---|---|
| AA composition | Frequency of each of 20 amino acids | First 20 aa |
| Hydrophobicity | Max and avg (Kyte-Doolittle, window=5) | First 40 aa |
| Charge | Max abundance of K/R and its normalised position (window=3) | First 40 aa |
| α-helix propensity | Max and avg (Chou-Fasman, window=7) | First 40 aa |
| TM propensity | Max and avg (Kyte scale, window=7) | First 40 aa |

**Total: 28 features** (20 composition + 2 hydrophobicity + 2 charge + 2 α-helix + 2 TM)

### Input files
- `positive.fasta`, `negative.fasta`
- `training_with_folds.tsv`, `benchmarking_set.tsv`

### Output files
- `figures/svm_*.pdf/.png` — CV curves, confusion matrix, feature importance
- Console summary: CV metrics + benchmark metrics

---
## Cell 1 — Install dependencies

In [ ]:
!pip install -q biopython pandas numpy scikit-learn matplotlib seaborn
print('Done.')

---
## Cell 2 — Imports, scales, and constants

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from Bio import SeqIO
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    precision_recall_curve, roc_curve, auc,
    matthews_corrcoef, f1_score, accuracy_score,
    precision_score, recall_score, confusion_matrix
)
from sklearn.model_selection import ParameterGrid
import warnings
warnings.filterwarnings('ignore')

os.makedirs('figures', exist_ok=True)
sns.set_theme(style='whitegrid', font_scale=1.1)

# --- Biochemical scales ---

# Kyte-Doolittle hydrophobicity
HYDROPATHY = {
    'A':  1.80, 'C':  2.50, 'D': -3.50, 'E': -3.50, 'F':  2.80,
    'G': -0.40, 'H': -3.20, 'I':  4.50, 'K': -3.90, 'L':  3.80,
    'M':  1.90, 'N': -3.50, 'P': -1.60, 'Q': -3.50, 'R': -4.50,
    'S': -0.80, 'T': -0.70, 'V':  4.20, 'W': -0.90, 'Y': -1.30
}

# Positive charge indicator (K, R = 1, others = 0)
CHARGE = {
    'A': 0, 'C': 0, 'D': 0, 'E': 0, 'F': 0,
    'G': 0, 'H': 0, 'I': 0, 'K': 1, 'L': 0,
    'M': 0, 'N': 0, 'P': 0, 'Q': 0, 'R': 1,
    'S': 0, 'T': 0, 'V': 0, 'W': 0, 'Y': 0
}

# Chou-Fasman alpha-helix propensity
ALPHA_HELIX = {
    'A': 1.42, 'C': 0.70, 'D': 1.01, 'E': 1.51, 'F': 1.13,
    'G': 0.57, 'H': 1.00, 'I': 1.08, 'K': 1.16, 'L': 1.21,
    'M': 1.45, 'N': 0.67, 'P': 0.57, 'Q': 1.11, 'R': 0.98,
    'S': 0.77, 'T': 0.83, 'V': 1.06, 'W': 1.08, 'Y': 0.69
}

# Transmembrane tendency (Kyte scale adapted)
TM_TENDENCY = {
    'A':  0.38, 'C': -0.30, 'D': -3.27, 'E': -2.90, 'F':  1.98,
    'G': -0.19, 'H': -1.44, 'I':  1.97, 'K': -3.46, 'L':  1.82,
    'M':  1.40, 'N': -1.62, 'P': -1.44, 'Q': -1.84, 'R': -2.57,
    'S': -0.53, 'T': -0.32, 'V':  1.46, 'W':  1.53, 'Y':  0.49
}

AA_LIST = sorted(HYDROPATHY.keys())  # 20 standard amino acids, alphabetical

print('Imports and scales loaded.')
print(f'Feature groups: AA composition (20) + hydrophobicity (2) + charge (2) + alpha-helix (2) + TM (2) = 28 total')

---
## Cell 3 — Feature extraction functions

In [ ]:
def sliding_window_values(values, window_size):
    """Return list of sliding window sums/averages over a list of values."""
    if len(values) < window_size:
        return []
    return [values[i:i+window_size] for i in range(len(values) - window_size + 1)]


def extract_features(seq):
    """
    Extract all 28 features from a protein sequence.

    All features are computed from the N-terminal region only,
    since signal peptides are always N-terminal.

    Returns a dict of feature_name -> float.
    """
    seq = seq.upper()
    features = {}

    # --- 1. Amino acid composition (first 20 aa) ---
    region = seq[:20]
    total  = len(region)
    for aa in AA_LIST:
        features[f'comp_{aa}'] = region.count(aa) / total if total > 0 else 0.0

    # --- 2. Hydrophobicity (first 40 aa, window=5) ---
    region40 = seq[:40]
    hydro_vals = [HYDROPATHY.get(aa, 0.0) for aa in region40]
    windows = sliding_window_values(hydro_vals, 5)
    if windows:
        window_sums = [sum(w) for w in windows]
        features['max_hydrophobicity'] = max(window_sums)
        features['avg_hydrophobicity'] = float(np.mean(window_sums))
    else:
        features['max_hydrophobicity'] = 0.0
        features['avg_hydrophobicity'] = 0.0

    # --- 3. Charge features (first 40 aa, window=3) ---
    charge_vals = [CHARGE.get(aa, 0) for aa in region40]
    windows = sliding_window_values(charge_vals, 3)
    if windows:
        window_sums = [sum(w) for w in windows]
        max_charge = max(window_sums)
        features['max_charge_abundance'] = float(max_charge)
        if max_charge > 0:
            max_idx = window_sums.index(max_charge)
            features['pos_max_charge'] = (max_idx + 1) / len(window_sums)
        else:
            features['pos_max_charge'] = 0.0
    else:
        features['max_charge_abundance'] = 0.0
        features['pos_max_charge'] = 0.0

    # --- 4. Alpha-helix propensity (first 40 aa, window=7) ---
    alpha_vals = [ALPHA_HELIX.get(aa, 0.0) for aa in region40]
    windows = sliding_window_values(alpha_vals, 7)
    if windows:
        window_avgs = [float(np.mean(w)) for w in windows]
        features['avg_alpha_propensity'] = float(np.mean(window_avgs))
        features['max_alpha_propensity'] = max(window_avgs)
    else:
        features['avg_alpha_propensity'] = 0.0
        features['max_alpha_propensity'] = 0.0

    # --- 5. Transmembrane propensity (first 40 aa, window=7) ---
    tm_vals = [TM_TENDENCY.get(aa, 0.0) for aa in region40]
    windows = sliding_window_values(tm_vals, 7)
    if windows:
        window_avgs = [float(np.mean(w)) for w in windows]
        features['avg_tm_propensity'] = float(np.mean(window_avgs))
        features['max_tm_propensity'] = max(window_avgs)
    else:
        features['avg_tm_propensity'] = 0.0
        features['max_tm_propensity'] = 0.0

    return features


# Quick sanity check
test_feat = extract_features('MKLVVVGAGGVGKSALTIQLIQNHFVDEYDPTIEDSY')
print(f'Features per sequence: {len(test_feat)}')
print(f'Feature names: {list(test_feat.keys())}')

---
## Cell 4 — Load data and build feature matrix

In [ ]:
# Load sequences
all_seqs = {}
for rec in SeqIO.parse('positive.fasta', 'fasta'):
    all_seqs[rec.id] = str(rec.seq)
for rec in SeqIO.parse('negative.fasta', 'fasta'):
    all_seqs[rec.id] = str(rec.seq)
print(f'Sequences loaded: {len(all_seqs):,}')

# Load metadata
df_train = pd.read_csv('training_with_folds.tsv', sep='\t')
df_bench = pd.read_csv('benchmarking_set.tsv',    sep='\t')

def build_feature_matrix(df, all_seqs):
    """Extract features for all sequences in df. Returns X (array), y (array), feature_names (list)."""
    records = []
    for _, row in df.iterrows():
        seq = all_seqs.get(row['Accession'], '')
        if not seq:
            continue
        feat = extract_features(seq)
        feat['label'] = row['label']
        feat['Accession'] = row['Accession']
        if 'fold' in df.columns:
            feat['fold'] = row['fold']
        records.append(feat)

    feat_df = pd.DataFrame(records)
    meta_cols = ['label', 'Accession'] + (['fold'] if 'fold' in feat_df.columns else [])
    feature_cols = [c for c in feat_df.columns if c not in meta_cols]
    return feat_df, feature_cols

print('Extracting training features...')
train_feat_df, FEATURE_COLS = build_feature_matrix(df_train, all_seqs)
print('Extracting benchmark features...')
bench_feat_df, _ = build_feature_matrix(df_bench, all_seqs)

print(f'\nTraining feature matrix : {train_feat_df.shape}')
print(f'Benchmark feature matrix: {bench_feat_df.shape}')
print(f'Number of features      : {len(FEATURE_COLS)}')

---
## Cell 5 — Feature selection via Random Forest importance

We use a Random Forest trained on each training fold to rank feature importance.
The average importance across all folds determines which features are selected.
This prevents data leakage — the feature selector only sees training data.

In [ ]:
folds = sorted(train_feat_df['fold'].unique())
importance_matrix = np.zeros((len(folds), len(FEATURE_COLS)))

for i, val_fold in enumerate(folds):
    cv_train = train_feat_df[train_feat_df['fold'] != val_fold]
    X_cv = cv_train[FEATURE_COLS].values
    y_cv = cv_train['label'].values
    rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
    rf.fit(X_cv, y_cv)
    importance_matrix[i] = rf.feature_importances_

mean_importance = importance_matrix.mean(axis=0)
importance_df = pd.DataFrame({
    'feature'   : FEATURE_COLS,
    'importance' : mean_importance
}).sort_values('importance', ascending=False).reset_index(drop=True)

# Select top 20 features
N_TOP = 20
TOP_FEATURES = importance_df['feature'].head(N_TOP).tolist()

print(f'Top {N_TOP} features by Random Forest importance:')
print(importance_df.head(N_TOP).to_string(index=False))

# Plot feature importances
fig, ax = plt.subplots(figsize=(10, 7))
top_df = importance_df.head(N_TOP)
ax.barh(top_df['feature'][::-1], top_df['importance'][::-1], color='steelblue', alpha=0.85)
ax.set_xlabel('Mean importance (across 5 folds)')
ax.set_title(f'Top {N_TOP} Features — Random Forest Importance', fontweight='bold')
plt.tight_layout()
plt.savefig('figures/svm_feature_importance.pdf', bbox_inches='tight')
plt.savefig('figures/svm_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: figures/svm_feature_importance.pdf/.png')

---
## Cell 6 — 5-fold CV with grid search (ALL features)

For each fold:
- Fit a `StandardScaler` on training folds only (no leakage)
- Grid search over C and γ using MCC as the optimisation metric
- Evaluate on the validation fold

In [ ]:
# Hyperparameter grid
PARAM_GRID = {
    'C'    : [0.1, 1, 10],
    'gamma': ['scale', 0.01, 0.001]
}

def run_svm_cv(train_feat_df, feature_cols, folds, param_grid):
    """
    Run 5-fold CV with nested grid search.
    Returns: all_scores, all_labels, all_preds, best_params_per_fold
    """
    all_scores, all_labels, all_preds = [], [], []
    best_params_list = []

    print(f'{"Fold":<6} {"Best C":<10} {"Best γ":<12} {"Val MCC":<10} {"Val F1"}')
    print('-' * 50)

    for val_fold in folds:
        df_cv_train = train_feat_df[train_feat_df['fold'] != val_fold]
        df_cv_val   = train_feat_df[train_feat_df['fold'] == val_fold]

        X_tr = df_cv_train[feature_cols].values
        y_tr = df_cv_train['label'].values
        X_val = df_cv_val[feature_cols].values
        y_val = df_cv_val['label'].values

        # Fit scaler on training folds only
        scaler = StandardScaler()
        X_tr_scaled  = scaler.fit_transform(X_tr)
        X_val_scaled = scaler.transform(X_val)

        # Grid search: pick params with best MCC on validation fold
        best_mcc, best_params, best_model = -2, None, None
        for params in ParameterGrid(param_grid):
            svm = SVC(kernel='rbf', probability=True, random_state=42, **params)
            svm.fit(X_tr_scaled, y_tr)
            preds = svm.predict(X_val_scaled)
            mcc = matthews_corrcoef(y_val, preds)
            if mcc > best_mcc:
                best_mcc    = mcc
                best_params = params
                best_model  = svm

        # Collect scores and predictions from best model
        fold_probs = best_model.predict_proba(X_val_scaled)[:, 1]
        fold_preds = best_model.predict(X_val_scaled)
        fold_f1    = f1_score(y_val, fold_preds)

        all_scores.extend(fold_probs)
        all_labels.extend(y_val)
        all_preds.extend(fold_preds)
        best_params_list.append(best_params)

        print(f"{val_fold:<6} {str(best_params['C']):<10} {str(best_params['gamma']):<12} "
              f"{best_mcc:<10.3f} {fold_f1:.3f}")

    return np.array(all_scores), np.array(all_labels), np.array(all_preds), best_params_list


print('=== SVM with ALL FEATURES ===')
scores_all, labels_all, preds_all, params_all = run_svm_cv(
    train_feat_df, FEATURE_COLS, folds, PARAM_GRID
)

cv_metrics_all = {
    'MCC'      : matthews_corrcoef(labels_all, preds_all),
    'Precision': precision_score(labels_all, preds_all, zero_division=0),
    'Recall'   : recall_score(labels_all, preds_all, zero_division=0),
    'F1'       : f1_score(labels_all, preds_all, zero_division=0),
    'Accuracy' : accuracy_score(labels_all, preds_all),
}
print('\nCV metrics (all features):')
for k, v in cv_metrics_all.items():
    print(f'  {k:<12}: {v:.3f}')

---
## Cell 7 — 5-fold CV with grid search (SELECTED features)

In [ ]:
print('=== SVM with SELECTED FEATURES (top 20) ===')
scores_sel, labels_sel, preds_sel, params_sel = run_svm_cv(
    train_feat_df, TOP_FEATURES, folds, PARAM_GRID
)

cv_metrics_sel = {
    'MCC'      : matthews_corrcoef(labels_sel, preds_sel),
    'Precision': precision_score(labels_sel, preds_sel, zero_division=0),
    'Recall'   : recall_score(labels_sel, preds_sel, zero_division=0),
    'F1'       : f1_score(labels_sel, preds_sel, zero_division=0),
    'Accuracy' : accuracy_score(labels_sel, preds_sel),
}
print('\nCV metrics (selected features):')
for k, v in cv_metrics_sel.items():
    print(f'  {k:<12}: {v:.3f}')

---
## Cell 8 — CV curves (PR + ROC) for all-feature model

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PR curve
prec, rec, _ = precision_recall_curve(labels_all, scores_all)
pr_auc = auc(rec, prec)
ax = axes[0]
ax.plot(rec, prec, color='steelblue', linewidth=1.8, label=f'PR curve (AUC={pr_auc:.3f})')
ax.scatter([cv_metrics_all['Recall']], [cv_metrics_all['Precision']],
           color='red', s=60, zorder=5,
           label=f'Operating point (F1={cv_metrics_all["F1"]:.3f})')
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_title('CV PR Curve (all features)')
ax.legend(); ax.set_xlim(0, 1.02); ax.set_ylim(0, 1.05)

# ROC curve
fpr, tpr, _ = roc_curve(labels_all, scores_all)
roc_auc = auc(fpr, tpr)
ax2 = axes[1]
ax2.plot(fpr, tpr, color='purple', linewidth=1.8, label=f'ROC curve (AUC={roc_auc:.3f})')
ax2.plot([0,1],[0,1],'k--', linewidth=1, label='Random')
ax2.set_xlabel('False Positive Rate'); ax2.set_ylabel('True Positive Rate')
ax2.set_title('CV ROC Curve (all features)')
ax2.legend(); ax2.set_xlim(0, 1.02); ax2.set_ylim(0, 1.05)

plt.suptitle('SVM Classifier — 5-Fold Cross-Validation (All Features)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/svm_cv_pr_roc_all.pdf', bbox_inches='tight')
plt.savefig('figures/svm_cv_pr_roc_all.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'PR AUC={pr_auc:.3f}  |  ROC AUC={roc_auc:.3f}')
print('Saved: figures/svm_cv_pr_roc_all.pdf/.png')

---
## Cell 9 — Train final model on full training set + benchmark evaluation

We pick the most common best hyperparameters across folds,
retrain on the **entire** training set, and evaluate on the blind benchmark.

In [ ]:
from collections import Counter

def train_and_benchmark(train_feat_df, bench_feat_df, feature_cols, params_list, label):
    """
    Train final SVM on full training set with most-common CV hyperparameters.
    Evaluate on benchmark set. Returns benchmark metrics dict.
    """
    # Pick most common (C, gamma) pair from CV folds
    param_tuples = [(p['C'], str(p['gamma'])) for p in params_list]
    best_C, best_gamma_str = Counter(param_tuples).most_common(1)[0][0]
    best_gamma = float(best_gamma_str) if best_gamma_str not in ('scale', 'auto') else best_gamma_str
    print(f'[{label}] Final hyperparameters: C={best_C}, gamma={best_gamma}')

    # Prepare full training data
    X_train = train_feat_df[feature_cols].values
    y_train = train_feat_df['label'].values
    X_bench = bench_feat_df[feature_cols].values
    y_bench = bench_feat_df['label'].values

    # Scale (fit on training only)
    scaler = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train)
    X_bench_sc = scaler.transform(X_bench)

    # Train final model
    svm_final = SVC(kernel='rbf', C=best_C, gamma=best_gamma,
                    probability=True, random_state=42)
    svm_final.fit(X_train_sc, y_train)

    # Benchmark evaluation
    bench_probs = svm_final.predict_proba(X_bench_sc)[:, 1]
    bench_preds = svm_final.predict(X_bench_sc)

    metrics = {
        'MCC'      : matthews_corrcoef(y_bench, bench_preds),
        'Precision': precision_score(y_bench, bench_preds, zero_division=0),
        'Recall'   : recall_score(y_bench, bench_preds, zero_division=0),
        'F1'       : f1_score(y_bench, bench_preds, zero_division=0),
        'Accuracy' : accuracy_score(y_bench, bench_preds),
    }
    return metrics, bench_probs, bench_preds, y_bench, svm_final, scaler


# All features
bench_metrics_all, bench_probs_all, bench_preds_all, y_bench, model_all, scaler_all = \
    train_and_benchmark(train_feat_df, bench_feat_df, FEATURE_COLS, params_all, 'ALL')

print('\nBenchmark metrics (all features):')
for k, v in bench_metrics_all.items():
    print(f'  {k:<12}: {v:.3f}')

# Selected features
bench_metrics_sel, bench_probs_sel, bench_preds_sel, _, model_sel, scaler_sel = \
    train_and_benchmark(train_feat_df, bench_feat_df, TOP_FEATURES, params_sel, 'SELECTED')

print('\nBenchmark metrics (selected features):')
for k, v in bench_metrics_sel.items():
    print(f'  {k:<12}: {v:.3f}')

---
## Cell 10 — Benchmark confusion matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, preds, title in zip(axes,
    [bench_preds_all, bench_preds_sel],
    ['All features (28)', f'Selected features ({N_TOP})']):

    cm = confusion_matrix(y_bench, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Pred SP−', 'Pred SP+'],
                yticklabels=['True SP−', 'True SP+'],
                linewidths=0.5, cbar=False)
    tn, fp, fn, tp = cm.ravel()
    ax.set_title(f'Confusion Matrix — {title}\nTP={tp} FP={fp} TN={tn} FN={fn}',
                 fontsize=11, fontweight='bold')

plt.suptitle('SVM Classifier — Benchmark Set Confusion Matrices', fontsize=13)
plt.tight_layout()
plt.savefig('figures/svm_benchmark_confusion.pdf', bbox_inches='tight')
plt.savefig('figures/svm_benchmark_confusion.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: figures/svm_benchmark_confusion.pdf/.png')

---
## Cell 11 — Final summary table

In [ ]:
sep = '=' * 66
print(sep)
print('  SVM CLASSIFIER SUMMARY — LB2 Group 7')
print(sep)
print()
print(f'  {"Metric":<14} {"CV (all)":>10} {"Bench (all)":>13} {"CV (sel)":>10} {"Bench (sel)":>13}')
print('  ' + '-' * 62)
for metric in ['MCC', 'Precision', 'Recall', 'F1', 'Accuracy']:
    print(f'  {metric:<14} '
          f'{cv_metrics_all[metric]:>10.3f} '
          f'{bench_metrics_all[metric]:>13.3f} '
          f'{cv_metrics_sel[metric]:>10.3f} '
          f'{bench_metrics_sel[metric]:>13.3f}')
print()
print('  Feature sets')
print(f'    All features      : {len(FEATURE_COLS)}')
print(f'    Selected features : {N_TOP} (by Random Forest importance)')
print(f'    Top selected      : {TOP_FEATURES[:5]} ...')
print()
print('  Hyperparameter search')
print(f'    Kernel  : RBF')
print(f'    C grid  : {PARAM_GRID["C"]}')
print(f'    γ grid  : {PARAM_GRID["gamma"]}')
print(f'    Metric  : MCC (maximised)')
print()
print('  Output figures')
for f in sorted(os.listdir('figures')):
    if f.startswith('svm_'):
        print(f'    figures/{f}')
print()
print(sep)